# Measure a synthetic company CSV locally
By **Pablo Cirre — CompaniesData.cloud**. Python 3.10+, standard library only.

All records in this demonstration are fictional. Domains use `.example`, phones are empty, and each row is marked `synthetic=true`. Code and narrative: MIT. Fixture data and their aggregate outputs: CC0 1.0.

Run from the kit directory or its notebooks directory. No files are uploaded and no network request is made. Passing syntax does not establish company correctness, coverage, consent, or email deliverability.

In [1]:
from pathlib import Path
import json
import sys

kit_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'company_data_quality').is_dir() and (p / 'fixtures').is_dir()), None)
assert kit_root is not None, 'Open this notebook from the kit or notebooks directory'
sys.path.insert(0, str(kit_root))
from company_data_quality import validate_csv, report_csv
required = ['company_id', 'company_name', 'country_code']
formats = {'country_code': 'country_code_alpha2', 'website': 'url_http', 'email': 'email_ascii', 'observed_at': 'date_iso'}
report = validate_csv(kit_root / 'fixtures' / 'synthetic_company_data_comma.csv', required_fields=required, formats=formats)
assert report['status'] == 'needs_review'
print(json.dumps(report['summary'], indent=2))


{
  "data_records": 12,
  "column_count": 8,
  "blank_records_skipped": 0,
  "required_column_count": 3,
  "format_rule_count": 4,
  "required_complete_records": 9,
  "missing_required_cells": 3,
  "format_failed_cells": 8,
  "exact_duplicate_records_beyond_first": 1,
  "exact_duplicate_groups": 1
}


## Separate the denominators
There are 12 logical records, not 12 physical lines: a quoted company label contains a newline. There are 3 missing required cells and 8 failed format checks. Do not add these into a count of bad companies because one record may have several findings.

Company-name completeness is 10/12. Email syntax is checked only for 11 non-empty cells; 3 fail and 8 pass the limited rule. None of those 8 fictional mailbox strings is certified deliverable.

In [2]:
summary = report['summary']
assert summary['data_records'] == 12
assert summary['missing_required_cells'] == 3
assert summary['format_failed_cells'] == 8
assert summary['required_complete_records'] == 9
assert summary['exact_duplicate_records_beyond_first'] == 1
name, email = report['columns'][1], report['columns'][4]
assert name['completeness_pct'] == 83.33
assert (email['present'], email['format_passed'], email['format_failed']) == (11, 8, 3)
print(json.dumps({'company_name_column': name, 'email_column': email}, indent=2))


{
  "company_name_column": {
    "column_id": "column_2",
    "required": true,
    "format_rule": null,
    "present": 10,
    "missing": 2,
    "completeness_pct": 83.33,
    "format_checked": 0,
    "format_passed": 0,
    "format_failed": 0
  },
  "email_column": {
    "column_id": "column_5",
    "required": false,
    "format_rule": "email_ascii",
    "present": 11,
    "missing": 1,
    "completeness_pct": 91.67,
    "format_checked": 11,
    "format_passed": 8,
    "format_failed": 3
  }
}


## Change the transport, not the meaning
The second fixture contains the same fields and records, separated by semicolons and encoded with a UTF-8 BOM. Identical aggregate results demonstrate that the reader handles this example consistently. Ragged records and malformed CSV fail without a partial quality report.

In [3]:
other = validate_csv(kit_root / 'fixtures' / 'synthetic_company_data_semicolon_bom.csv', required_fields=required, formats=formats)
assert other['summary'] == report['summary']
assert other['columns'] == report['columns']
assert other['input_format']['delimiter'] == 'semicolon'
print('PASS: comma and semicolon+BOM produce the same aggregate measurements.')


PASS: comma and semicolon+BOM produce the same aggregate measurements.


## Export counts, then make a procurement decision
The flat CSV report below uses positions rather than original header names. It contains no source row values. Aggregation alone is not a guarantee of anonymity, especially for very small populations. Review before sharing.

Exact duplicates compare all parsed values without normalizing case or whitespace. They do not identify two different rows describing the same company. Keep completeness, syntax, entity matching, sample representativeness, rights, and market coverage as separate questions. Use the buyer checklist to document the evidence needed for each.

In [4]:
csv_report = report_csv(report)
assert 'contact@' not in csv_report
assert 'company-001.example' not in csv_report
print('\n'.join(csv_report.splitlines()[:12]))
print('No real company or contact records were used in this notebook.')


section,column_id,metric,value
report,,status,needs_review
summary,,data_records,12
summary,,column_count,8
summary,,blank_records_skipped,0
summary,,required_column_count,3
summary,,format_rule_count,4
summary,,required_complete_records,9
summary,,missing_required_cells,3
summary,,format_failed_cells,8
summary,,exact_duplicate_records_beyond_first,1
summary,,exact_duplicate_groups,1
No real company or contact records were used in this notebook.
